In [1]:
import cobra
import pandas as pd
import os
from os.path import join
from cobra.io import load_matlab_model

output_dir = '/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Pediatric_pfba3/'
fractions = [0.005,0.002]#pfba3 folder
# experimental oxygen consumption rate defined here
mean_ocr = 0.286
sd_ocr = 0.093
max_ocr = mean_ocr+sd_ocr
min_ocr=mean_ocr-sd_ocr
# GEMs have either one of the two cytochrome oxidase reactions, depending on the gene expression data
primary_rxn = "CYOOm3i"
fallback_rxn = "CYOOm2i"
# List of model filenames
core_models=['ACH-000080',
'ACH-000045',
'ACH-000146',
'ACH-000263',
'ACH-000602',
'ACH-000770',
'ACH-000557',
'ACH-000641',
'ACH-001036',
'ACH-000020',
'ACH-000032',
'ACH-000059',
'ACH-000070',
'ACH-000151',
'ACH-000156',
'ACH-000728',
'ACH-000782',
'ACH-000922',
'ACH-000960',
'ACH-001106',
'ACH-001669',
'ACH-001735',
'ACH-001736',
'ACH-001993',
'ACH-002059',
'ACH-000160',
'ACH-001020',
'ACH-001028',
'ACH-001031',
'ACH-001289',
'ACH-000055',
'ACH-000095',
'ACH-000211',
'ACH-000776',
'ACH-001053',
'ACH-001054',
'ACH-001201',
'ACH-001232',
'ACH-001033',
'ACH-001711',
'ACH-000039',
'ACH-000052',
'ACH-000087',
'ACH-000279',
'ACH-000391',
'ACH-000499',
'ACH-001029',
'ACH-001032',
'ACH-001034',
'ACH-001035',
'ACH-001038',
'ACH-001193',
'ACH-001430',
'ACH-001431',
'ACH-000245',
'ACH-000402',
'ACH-000707',
'ACH-000786',
'ACH-000877',
'ACH-000944',
'ACH-001064',
'ACH-002055',
'ACH-000660',
'ACH-000099',
'ACH-000120',
'ACH-000136',
'ACH-000149',
'ACH-000203',
'ACH-000227',
'ACH-000259',
'ACH-000260',
'ACH-000310',
'ACH-000312',
'ACH-000341',
'ACH-000345',
'ACH-000366',
'ACH-000446',
'ACH-000804',
'ACH-001188',
'ACH-001300',
'ACH-001301',
'ACH-001302',
'ACH-001303',
'ACH-001338',
'ACH-001344',
'ACH-001354',
'ACH-001366',
'ACH-001367',
'ACH-001481',
'ACH-001548',
'ACH-001603',
'ACH-001674',
'ACH-001716',
'ACH-002922',
'ACH-000082',
'ACH-000359',
'ACH-000364',
'ACH-000410',
'ACH-000613',
'ACH-001001',
'ACH-001526',
'ACH-001715',
'ACH-001814',
'ACH-002067',
'ACH-002069',
'ACH-002471',
'ACH-002834',
'ACH-000597',
'ACH-001059',
'ACH-001099',
'ACH-001128',
'ACH-001211',
'ACH-000096',
'ACH-000201',
'ACH-000375',
'ACH-000533',
'ACH-000607',
'ACH-001109',
'ACH-001200',
'ACH-001210',
'ACH-001532',
'ACH-000833',
'ACH-001050',
'ACH-001096',
'ACH-001184',
'ACH-001740',
'ACH-001743',
'ACH-001745',
'ACH-001765',
'ACH-000169',
'ACH-000689',
'ACH-001196',
'ACH-001750',
'ACH-001751',
'ACH-002048',
'ACH-000051',
'ACH-000105',
'ACH-000372',
'ACH-000519',
'ACH-000636',
'ACH-000918',
'ACH-000937',
'ACH-000942',
'ACH-000953',
'ACH-000981',
'ACH-000995',
'ACH-001737']  # add all 147 models here

#
for idx, model_name in enumerate(core_models):
    print(f'\n=== MODEL {idx+1}/{len(core_models)}: {model_name} ===', flush=True)
    # Load the MATLAB model
    core_model = load_matlab_model(join(r'/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Pediatric_rsnew/', model_name))
    # Set solver timeout (seconds)
    core_model.solver.configuration.timeout = 500  # adjust as needed

    skip_model = False

    # Get reaction names
    rxn_names = [r.name for r in core_model.reactions]
    output_path = os.path.join(output_dir, f"{model_name}_pfba.xlsx")
    # Open Excel writer
    with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
        for j, f in enumerate(fractions):
            # Set bounds
            if primary_rxn in core_model.reactions:
                core_model.reactions.get_by_id(primary_rxn).bounds = (f * min_ocr, f * max_ocr)
            else:
                core_model.reactions.get_by_id(fallback_rxn).bounds = (f * min_ocr, f * max_ocr)
            core_model.reactions.get_by_id('EX_o2[e]').bounds = (f * -1.19, 0) # experimental oxygen uptake rate defined

            # FEASIBILITY CHECK
            print(f'Checking feasibility: Model {idx+1}, fraction {f}', flush=True)
            try:
                solution = core_model.optimize()
            except Exception:
                print(f'  SOLVER TIMEOUT at feasibility check, fraction {f}')
                pd.DataFrame({'status': ['TIMEOUT']}).to_excel(
                    writer, sheet_name=f'{int(f*100)}%', index=False
                )
                continue  # skip this fraction

            # If infeasible
            if solution.status != 'optimal':
                if j == 0:
                    print(f'  INFEASIBLE at fraction {f}, skipping model')
                    skip_model = True
                    break 
                else:
                    print(f'  INFEASIBLE at fraction {f}, skipping fraction')
                    pd.DataFrame({'status': ['INFEASIBLE']}).to_excel(
                        writer, sheet_name=f'{int(f*100)}%', index=False
                    )
                    continue

            # pFBA
            print(f'Running pFBA: Model {idx+1}, fraction {f}', flush=True)
            try:
                pfba_solution = cobra.flux_analysis.pfba(core_model)
            except Exception:
                print(f'  SOLVER TIMEOUT at pFBA, fraction {f}, skipping fraction')
                pd.DataFrame({'status': ['TIMEOUT']}).to_excel(
                    writer, sheet_name=f'{float(f)}', index=False
                )
                continue

            # dataframe
            pfba_df = pd.DataFrame({
                'Reactions': [r.id for r in core_model.reactions],
                'fluxes': pfba_solution.fluxes,
                'names': rxn_names
            })
            pfba_df = pfba_df[abs(pfba_df.fluxes) > core_model.tolerance]
            # save the sheet name as below
            sheet_name = f'{float(f)}'
            pfba_df.to_excel(writer, sheet_name=sheet_name, index=False)

            print(f'Model {idx+1}, fraction {f}: {len(pfba_df)} non-zero fluxes, sum={pfba_df.fluxes.sum()}')

    # Move to next model if first fraction was infeasible
    if skip_model:
        continue



=== MODEL 1/147: ACH-000080 ===
Checking feasibility: Model 1, fraction 0.005
Running pFBA: Model 1, fraction 0.005
Model 1, fraction 0.005: 525 non-zero fluxes, sum=5.85339142204988
Checking feasibility: Model 1, fraction 0.002
Running pFBA: Model 1, fraction 0.002
Model 1, fraction 0.002: 518 non-zero fluxes, sum=5.248265142779442

=== MODEL 2/147: ACH-000045 ===
Checking feasibility: Model 2, fraction 0.005
Running pFBA: Model 2, fraction 0.005
Model 2, fraction 0.005: 303 non-zero fluxes, sum=2.1069049519324143
Checking feasibility: Model 2, fraction 0.002
Running pFBA: Model 2, fraction 0.002
Model 2, fraction 0.002: 310 non-zero fluxes, sum=0.9468649320926156

=== MODEL 3/147: ACH-000146 ===
Checking feasibility: Model 3, fraction 0.005
Running pFBA: Model 3, fraction 0.005
Model 3, fraction 0.005: 335 non-zero fluxes, sum=0.4052637024515621
Checking feasibility: Model 3, fraction 0.002
Running pFBA: Model 3, fraction 0.002
Model 3, fraction 0.002: 331 non-zero fluxes, sum=0.185

In [2]:
import cobra
import pandas as pd
import os
from os.path import join
from cobra.io import load_matlab_model

output_dir = '/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Pediatric_pfba2/'
fractions = [0.5,0.2]#pfba2 folder
mean_ocr = 0.286
sd_ocr = 0.093
max_ocr = mean_ocr+sd_ocr
min_ocr=mean_ocr-sd_ocr
primary_rxn = "CYOOm3i"
fallback_rxn = "CYOOm2i"
# List of model filenames
core_models=['ACH-000080',
'ACH-000045',
'ACH-000146',
'ACH-000263',
'ACH-000602',
'ACH-000770',
'ACH-000557',
'ACH-000641',
'ACH-001036',
'ACH-000020',
'ACH-000032',
'ACH-000059',
'ACH-000070',
'ACH-000151',
'ACH-000156',
'ACH-000728',
'ACH-000782',
'ACH-000922',
'ACH-000960',
'ACH-001106',
'ACH-001669',
'ACH-001735',
'ACH-001736',
'ACH-001993',
'ACH-002059',
'ACH-000160',
'ACH-001020',
'ACH-001028',
'ACH-001031',
'ACH-001289',
'ACH-000055',
'ACH-000095',
'ACH-000211',
'ACH-000776',
'ACH-001053',
'ACH-001054',
'ACH-001201',
'ACH-001232',
'ACH-001033',
'ACH-001711',
'ACH-000039',
'ACH-000052',
'ACH-000087',
'ACH-000279',
'ACH-000391',
'ACH-000499',
'ACH-001029',
'ACH-001032',
'ACH-001034',
'ACH-001035',
'ACH-001038',
'ACH-001193',
'ACH-001430',
'ACH-001431',
'ACH-000245',
'ACH-000402',
'ACH-000707',
'ACH-000786',
'ACH-000877',
'ACH-000944',
'ACH-001064',
'ACH-002055',
'ACH-000660',
'ACH-000099',
'ACH-000120',
'ACH-000136',
'ACH-000149',
'ACH-000203',
'ACH-000227',
'ACH-000259',
'ACH-000260',
'ACH-000310',
'ACH-000312',
'ACH-000341',
'ACH-000345',
'ACH-000366',
'ACH-000446',
'ACH-000804',
'ACH-001188',
'ACH-001300',
'ACH-001301',
'ACH-001302',
'ACH-001303',
'ACH-001338',
'ACH-001344',
'ACH-001354',
'ACH-001366',
'ACH-001367',
'ACH-001481',
'ACH-001548',
'ACH-001603',
'ACH-001674',
'ACH-001716',
'ACH-002922',
'ACH-000082',
'ACH-000359',
'ACH-000364',
'ACH-000410',
'ACH-000613',
'ACH-001001',
'ACH-001526',
'ACH-001715',
'ACH-001814',
'ACH-002067',
'ACH-002069',
'ACH-002471',
'ACH-002834',
'ACH-000597',
'ACH-001059',
'ACH-001099',
'ACH-001128',
'ACH-001211',
'ACH-000096',
'ACH-000201',
'ACH-000375',
'ACH-000533',
'ACH-000607',
'ACH-001109',
'ACH-001200',
'ACH-001210',
'ACH-001532',
'ACH-000833',
'ACH-001050',
'ACH-001096',
'ACH-001184',
'ACH-001740',
'ACH-001743',
'ACH-001745',
'ACH-001765',
'ACH-000169',
'ACH-000689',
'ACH-001196',
'ACH-001750',
'ACH-001751',
'ACH-002048',
'ACH-000051',
'ACH-000105',
'ACH-000372',
'ACH-000519',
'ACH-000636',
'ACH-000918',
'ACH-000937',
'ACH-000942',
'ACH-000953',
'ACH-000981',
'ACH-000995',
'ACH-001737']  # add all 147 models here

#
for idx, model_name in enumerate(core_models):
    print(f'\n=== MODEL {idx+1}/{len(core_models)}: {model_name} ===', flush=True)
    # Load the MATLAB model
    core_model = load_matlab_model(join(r'/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Pediatric_rsnew/', model_name))

    # Set solver timeout (seconds)
    core_model.solver.configuration.timeout = 500  # adjust as needed

    skip_model = False

    # Get reaction names
    rxn_names = [r.name for r in core_model.reactions]
    output_path = os.path.join(output_dir, f"{model_name}_pfba.xlsx")
    # Open Excel writer
    with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
        for j, f in enumerate(fractions):
            # Set bounds
            if primary_rxn in core_model.reactions:
                core_model.reactions.get_by_id(primary_rxn).bounds = (f * min_ocr, f * max_ocr)
            else:
                core_model.reactions.get_by_id(fallback_rxn).bounds = (f * min_ocr, f * max_ocr)
            core_model.reactions.get_by_id('EX_o2[e]').bounds = (f * -1.19, 0)

            # FEASIBILITY CHECK
            print(f'Checking feasibility: Model {idx+1}, fraction {f}', flush=True)
            try:
                solution = core_model.optimize()
            except Exception:
                print(f'  SOLVER TIMEOUT at feasibility check, fraction {f}')
                pd.DataFrame({'status': ['TIMEOUT']}).to_excel(
                    writer, sheet_name=f'{int(f*100)}%', index=False
                )
                continue  # skip this fraction

            # If infeasible
            if solution.status != 'optimal':
                if j == 0:
                    print(f'  INFEASIBLE at fraction {f}, skipping model')
                    skip_model = True
                    break 
                else:
                    print(f'  INFEASIBLE at fraction {f}, skipping fraction')
                    pd.DataFrame({'status': ['INFEASIBLE']}).to_excel(
                        writer, sheet_name=f'{int(f*100)}%', index=False
                    )
                    continue

            # pFBA
            print(f'Running pFBA: Model {idx+1}, fraction {f}', flush=True)
            try:
                pfba_solution = cobra.flux_analysis.pfba(core_model)
            except Exception:
                print(f'  SOLVER TIMEOUT at pFBA, fraction {f}, skipping fraction')
                pd.DataFrame({'status': ['TIMEOUT']}).to_excel(
                    writer, sheet_name=f'{float(f)}', index=False
                )
                continue

            # dataframe
            pfba_df = pd.DataFrame({
                'Reactions': [r.id for r in core_model.reactions],
                'fluxes': pfba_solution.fluxes,
                'names': rxn_names
            })
            pfba_df = pfba_df[abs(pfba_df.fluxes) > core_model.tolerance]

            sheet_name = f'{int(f*100)}%'
            pfba_df.to_excel(writer, sheet_name=sheet_name, index=False)

            print(f'Model {idx+1}, fraction {f}: {len(pfba_df)} non-zero fluxes, sum={pfba_df.fluxes.sum()}')

    # Move to next model if first fraction was infeasible
    if skip_model:
        continue



=== MODEL 1/147: ACH-000080 ===
Checking feasibility: Model 1, fraction 0.5
Running pFBA: Model 1, fraction 0.5
Model 1, fraction 0.5: 502 non-zero fluxes, sum=6.90429567520153
Checking feasibility: Model 1, fraction 0.2
Running pFBA: Model 1, fraction 0.2
Model 1, fraction 0.2: 474 non-zero fluxes, sum=5.244334441435775

=== MODEL 2/147: ACH-000045 ===
Checking feasibility: Model 2, fraction 0.5
Running pFBA: Model 2, fraction 0.5
Model 2, fraction 0.5: 251 non-zero fluxes, sum=3.9125164725085764
Checking feasibility: Model 2, fraction 0.2
Running pFBA: Model 2, fraction 0.2
Model 2, fraction 0.2: 253 non-zero fluxes, sum=3.0433654106709547

=== MODEL 3/147: ACH-000146 ===
Checking feasibility: Model 3, fraction 0.5
Running pFBA: Model 3, fraction 0.5
Model 3, fraction 0.5: 323 non-zero fluxes, sum=4.363757641933804
Checking feasibility: Model 3, fraction 0.2
Running pFBA: Model 3, fraction 0.2
Model 3, fraction 0.2: 323 non-zero fluxes, sum=3.3771150496377316

=== MODEL 4/147: ACH-0

In [3]:
import cobra
import pandas as pd
import os
from os.path import join
from cobra.io import load_matlab_model

output_dir = '/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Pediatric_pfba1/'
fractions = [0.05,0.02]#pfba1 folder
mean_ocr = 0.286
sd_ocr = 0.093
max_ocr = mean_ocr+sd_ocr
min_ocr=mean_ocr-sd_ocr
primary_rxn = "CYOOm3i"
fallback_rxn = "CYOOm2i"
# List of model filenames
core_models=['ACH-000080',
'ACH-000045',
'ACH-000146',
'ACH-000263',
'ACH-000602',
'ACH-000770',
'ACH-000557',
'ACH-000641',
'ACH-001036',
'ACH-000020',
'ACH-000032',
'ACH-000059',
'ACH-000070',
'ACH-000151',
'ACH-000156',
'ACH-000728',
'ACH-000782',
'ACH-000922',
'ACH-000960',
'ACH-001106',
'ACH-001669',
'ACH-001735',
'ACH-001736',
'ACH-001993',
'ACH-002059',
'ACH-000160',
'ACH-001020',
'ACH-001028',
'ACH-001031',
'ACH-001289',
'ACH-000055',
'ACH-000095',
'ACH-000211',
'ACH-000776',
'ACH-001053',
'ACH-001054',
'ACH-001201',
'ACH-001232',
'ACH-001033',
'ACH-001711',
'ACH-000039',
'ACH-000052',
'ACH-000087',
'ACH-000279',
'ACH-000391',
'ACH-000499',
'ACH-001029',
'ACH-001032',
'ACH-001034',
'ACH-001035',
'ACH-001038',
'ACH-001193',
'ACH-001430',
'ACH-001431',
'ACH-000245',
'ACH-000402',
'ACH-000707',
'ACH-000786',
'ACH-000877',
'ACH-000944',
'ACH-001064',
'ACH-002055',
'ACH-000660',
'ACH-000099',
'ACH-000120',
'ACH-000136',
'ACH-000149',
'ACH-000203',
'ACH-000227',
'ACH-000259',
'ACH-000260',
'ACH-000310',
'ACH-000312',
'ACH-000341',
'ACH-000345',
'ACH-000366',
'ACH-000446',
'ACH-000804',
'ACH-001188',
'ACH-001300',
'ACH-001301',
'ACH-001302',
'ACH-001303',
'ACH-001338',
'ACH-001344',
'ACH-001354',
'ACH-001366',
'ACH-001367',
'ACH-001481',
'ACH-001548',
'ACH-001603',
'ACH-001674',
'ACH-001716',
'ACH-002922',
'ACH-000082',
'ACH-000359',
'ACH-000364',
'ACH-000410',
'ACH-000613',
'ACH-001001',
'ACH-001526',
'ACH-001715',
'ACH-001814',
'ACH-002067',
'ACH-002069',
'ACH-002471',
'ACH-002834',
'ACH-000597',
'ACH-001059',
'ACH-001099',
'ACH-001128',
'ACH-001211',
'ACH-000096',
'ACH-000201',
'ACH-000375',
'ACH-000533',
'ACH-000607',
'ACH-001109',
'ACH-001200',
'ACH-001210',
'ACH-001532',
'ACH-000833',
'ACH-001050',
'ACH-001096',
'ACH-001184',
'ACH-001740',
'ACH-001743',
'ACH-001745',
'ACH-001765',
'ACH-000169',
'ACH-000689',
'ACH-001196',
'ACH-001750',
'ACH-001751',
'ACH-002048',
'ACH-000051',
'ACH-000105',
'ACH-000372',
'ACH-000519',
'ACH-000636',
'ACH-000918',
'ACH-000937',
'ACH-000942',
'ACH-000953',
'ACH-000981',
'ACH-000995',
'ACH-001737']  # add all 147 models here

#
for idx, model_name in enumerate(core_models):
    print(f'\n=== MODEL {idx+1}/{len(core_models)}: {model_name} ===', flush=True)
    # Load the MATLAB model
    core_model = load_matlab_model(join(r'/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Pediatric_rsnew/', model_name))

    # Set solver timeout (seconds)
    core_model.solver.configuration.timeout = 500  # adjust as needed

    skip_model = False

    # Get reaction names
    rxn_names = [r.name for r in core_model.reactions]
    output_path = os.path.join(output_dir, f"{model_name}_pfba.xlsx")
    # Open Excel writer
    with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
        for j, f in enumerate(fractions):
            # Set bounds
            if primary_rxn in core_model.reactions:
                core_model.reactions.get_by_id(primary_rxn).bounds = (f * min_ocr, f * max_ocr)
            else:
                core_model.reactions.get_by_id(fallback_rxn).bounds = (f * min_ocr, f * max_ocr)
            core_model.reactions.get_by_id('EX_o2[e]').bounds = (f * -1.19, 0)

            # FEASIBILITY CHECK
            print(f'Checking feasibility: Model {idx+1}, fraction {f}', flush=True)
            try:
                solution = core_model.optimize()
            except Exception:
                print(f'  SOLVER TIMEOUT at feasibility check, fraction {f}')
                pd.DataFrame({'status': ['TIMEOUT']}).to_excel(
                    writer, sheet_name=f'{int(f*100)}%', index=False
                )
                continue  # skip this fraction

            # If infeasible
            if solution.status != 'optimal':
                if j == 0:
                    print(f'  INFEASIBLE at fraction {f}, skipping model')
                    skip_model = True
                    break 
                else:
                    print(f'  INFEASIBLE at fraction {f}, skipping fraction')
                    pd.DataFrame({'status': ['INFEASIBLE']}).to_excel(
                        writer, sheet_name=f'{int(f*100)}%', index=False
                    )
                    continue

            # pFBA
            print(f'Running pFBA: Model {idx+1}, fraction {f}', flush=True)
            try:
                pfba_solution = cobra.flux_analysis.pfba(core_model)
            except Exception:
                print(f'  SOLVER TIMEOUT at pFBA, fraction {f}, skipping fraction')
                pd.DataFrame({'status': ['TIMEOUT']}).to_excel(
                    writer, sheet_name=f'{float(f)}', index=False
                )
                continue

            # dataframe
            pfba_df = pd.DataFrame({
                'Reactions': [r.id for r in core_model.reactions],
                'fluxes': pfba_solution.fluxes,
                'names': rxn_names
            })
            pfba_df = pfba_df[abs(pfba_df.fluxes) > core_model.tolerance]

            sheet_name = f'{int(f*100)}%'
            pfba_df.to_excel(writer, sheet_name=sheet_name, index=False)

            print(f'Model {idx+1}, fraction {f}: {len(pfba_df)} non-zero fluxes, sum={pfba_df.fluxes.sum()}')

    # Move to next model if first fraction was infeasible
    if skip_model:
        continue



=== MODEL 1/147: ACH-000080 ===
Checking feasibility: Model 1, fraction 0.05
Running pFBA: Model 1, fraction 0.05
Model 1, fraction 0.05: 504 non-zero fluxes, sum=5.662375145972472
Checking feasibility: Model 1, fraction 0.02
Running pFBA: Model 1, fraction 0.02
Model 1, fraction 0.02: 504 non-zero fluxes, sum=5.303912495646165

=== MODEL 2/147: ACH-000045 ===
Checking feasibility: Model 2, fraction 0.05
Running pFBA: Model 2, fraction 0.05
Model 2, fraction 0.05: 258 non-zero fluxes, sum=2.767483215791984
Checking feasibility: Model 2, fraction 0.02
Running pFBA: Model 2, fraction 0.02
Model 2, fraction 0.02: 257 non-zero fluxes, sum=2.47933920069873

=== MODEL 3/147: ACH-000146 ===
Checking feasibility: Model 3, fraction 0.05
Running pFBA: Model 3, fraction 0.05
Model 3, fraction 0.05: 305 non-zero fluxes, sum=2.563320237856342
Checking feasibility: Model 3, fraction 0.02
Running pFBA: Model 3, fraction 0.02
Model 3, fraction 0.02: 327 non-zero fluxes, sum=1.8088491789717294

=== MO

In [4]:
import cobra
import pandas as pd
import os
from os.path import join
from cobra.io import load_matlab_model

output_dir = '/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Pediatric_pfba/'
fractions = [1,0.1,0.01,0.001]#pfba folder
mean_ocr = 0.286
sd_ocr = 0.093
max_ocr = mean_ocr+sd_ocr
min_ocr=mean_ocr-sd_ocr
primary_rxn = "CYOOm3i"
fallback_rxn = "CYOOm2i"
# List of model filenames
core_models=['ACH-000080',
'ACH-000045',
'ACH-000146',
'ACH-000263',
'ACH-000602',
'ACH-000770',
'ACH-000557',
'ACH-000641',
'ACH-001036',
'ACH-000020',
'ACH-000032',
'ACH-000059',
'ACH-000070',
'ACH-000151',
'ACH-000156',
'ACH-000728',
'ACH-000782',
'ACH-000922',
'ACH-000960',
'ACH-001106',
'ACH-001669',
'ACH-001735',
'ACH-001736',
'ACH-001993',
'ACH-002059',
'ACH-000160',
'ACH-001020',
'ACH-001028',
'ACH-001031',
'ACH-001289',
'ACH-000055',
'ACH-000095',
'ACH-000211',
'ACH-000776',
'ACH-001053',
'ACH-001054',
'ACH-001201',
'ACH-001232',
'ACH-001033',
'ACH-001711',
'ACH-000039',
'ACH-000052',
'ACH-000087',
'ACH-000279',
'ACH-000391',
'ACH-000499',
'ACH-001029',
'ACH-001032',
'ACH-001034',
'ACH-001035',
'ACH-001038',
'ACH-001193',
'ACH-001430',
'ACH-001431',
'ACH-000245',
'ACH-000402',
'ACH-000707',
'ACH-000786',
'ACH-000877',
'ACH-000944',
'ACH-001064',
'ACH-002055',
'ACH-000660',
'ACH-000099',
'ACH-000120',
'ACH-000136',
'ACH-000149',
'ACH-000203',
'ACH-000227',
'ACH-000259',
'ACH-000260',
'ACH-000310',
'ACH-000312',
'ACH-000341',
'ACH-000345',
'ACH-000366',
'ACH-000446',
'ACH-000804',
'ACH-001188',
'ACH-001300',
'ACH-001301',
'ACH-001302',
'ACH-001303',
'ACH-001338',
'ACH-001344',
'ACH-001354',
'ACH-001366',
'ACH-001367',
'ACH-001481',
'ACH-001548',
'ACH-001603',
'ACH-001674',
'ACH-001716',
'ACH-002922',
'ACH-000082',
'ACH-000359',
'ACH-000364',
'ACH-000410',
'ACH-000613',
'ACH-001001',
'ACH-001526',
'ACH-001715',
'ACH-001814',
'ACH-002067',
'ACH-002069',
'ACH-002471',
'ACH-002834',
'ACH-000597',
'ACH-001059',
'ACH-001099',
'ACH-001128',
'ACH-001211',
'ACH-000096',
'ACH-000201',
'ACH-000375',
'ACH-000533',
'ACH-000607',
'ACH-001109',
'ACH-001200',
'ACH-001210',
'ACH-001532',
'ACH-000833',
'ACH-001050',
'ACH-001096',
'ACH-001184',
'ACH-001740',
'ACH-001743',
'ACH-001745',
'ACH-001765',
'ACH-000169',
'ACH-000689',
'ACH-001196',
'ACH-001750',
'ACH-001751',
'ACH-002048',
'ACH-000051',
'ACH-000105',
'ACH-000372',
'ACH-000519',
'ACH-000636',
'ACH-000918',
'ACH-000937',
'ACH-000942',
'ACH-000953',
'ACH-000981',
'ACH-000995',
'ACH-001737']  # add all 147 models here

#
for idx, model_name in enumerate(core_models):
    print(f'\n=== MODEL {idx+1}/{len(core_models)}: {model_name} ===', flush=True)
    # Load the MATLAB model
    core_model = load_matlab_model(join(r'/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Pediatric_rsnew/', model_name))

    # Set solver timeout (seconds)
    core_model.solver.configuration.timeout = 500  # adjust as needed

    skip_model = False

    # Get reaction names
    rxn_names = [r.name for r in core_model.reactions]
    output_path = os.path.join(output_dir, f"{model_name}_pfba.xlsx")
    # Open Excel writer
    with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
        for j, f in enumerate(fractions):
            # Set bounds
            if primary_rxn in core_model.reactions:
                core_model.reactions.get_by_id(primary_rxn).bounds = (f * min_ocr, f * max_ocr)
            else:
                core_model.reactions.get_by_id(fallback_rxn).bounds = (f * min_ocr, f * max_ocr)
            core_model.reactions.get_by_id('EX_o2[e]').bounds = (f * -1.19, 0)

            # FEASIBILITY CHECK
            print(f'Checking feasibility: Model {idx+1}, fraction {f}', flush=True)
            try:
                solution = core_model.optimize()
            except Exception:
                print(f'  SOLVER TIMEOUT at feasibility check, fraction {f}')
                pd.DataFrame({'status': ['TIMEOUT']}).to_excel(
                    writer, sheet_name=f'{int(f*100)}%', index=False
                )
                continue  # skip this fraction

            # If infeasible
            if solution.status != 'optimal':
                if j == 0:
                    print(f'  INFEASIBLE at fraction {f}, skipping model')
                    skip_model = True
                    break 
                else:
                    print(f'  INFEASIBLE at fraction {f}, skipping fraction')
                    pd.DataFrame({'status': ['INFEASIBLE']}).to_excel(
                        writer, sheet_name=f'{int(f*100)}%', index=False
                    )
                    continue

            # pFBA
            print(f'Running pFBA: Model {idx+1}, fraction {f}', flush=True)
            try:
                pfba_solution = cobra.flux_analysis.pfba(core_model)
            except Exception:
                print(f'  SOLVER TIMEOUT at pFBA, fraction {f}, skipping fraction')
                pd.DataFrame({'status': ['TIMEOUT']}).to_excel(
                    writer, sheet_name=f'{float(f)}', index=False
                )
                continue

            # dataframe
            pfba_df = pd.DataFrame({
                'Reactions': [r.id for r in core_model.reactions],
                'fluxes': pfba_solution.fluxes,
                'names': rxn_names
            })
            pfba_df = pfba_df[abs(pfba_df.fluxes) > core_model.tolerance]

            sheet_name = f'{int(f*100)}%'
            pfba_df.to_excel(writer, sheet_name=sheet_name, index=False)

            print(f'Model {idx+1}, fraction {f}: {len(pfba_df)} non-zero fluxes, sum={pfba_df.fluxes.sum()}')

    # Move to next model if first fraction was infeasible
    if skip_model:
        continue



=== MODEL 1/147: ACH-000080 ===
Checking feasibility: Model 1, fraction 1
Running pFBA: Model 1, fraction 1
Model 1, fraction 1: 491 non-zero fluxes, sum=8.747047761048822
Checking feasibility: Model 1, fraction 0.1
Running pFBA: Model 1, fraction 0.1
Model 1, fraction 0.1: 488 non-zero fluxes, sum=4.954987723570837
Checking feasibility: Model 1, fraction 0.01
Running pFBA: Model 1, fraction 0.01
Model 1, fraction 0.01: 517 non-zero fluxes, sum=5.2694283142620915
Checking feasibility: Model 1, fraction 0.001
Running pFBA: Model 1, fraction 0.001
Model 1, fraction 0.001: 503 non-zero fluxes, sum=5.249167552128675

=== MODEL 2/147: ACH-000045 ===
Checking feasibility: Model 2, fraction 1
Running pFBA: Model 2, fraction 1
Model 2, fraction 1: 251 non-zero fluxes, sum=3.900444301560564
Checking feasibility: Model 2, fraction 0.1
Running pFBA: Model 2, fraction 0.1
Model 2, fraction 0.1: 256 non-zero fluxes, sum=2.919078548525408
Checking feasibility: Model 2, fraction 0.01
Running pFBA: M